# `parse_question` — Brique 2 : parser une question utilisateur

Référence : chapitre **`docs/06_question_parsing.md`** (Kezhan).

Module testé ici : [`src/docpipeline/question_parsing/question_parsing.py`](../src/docpipeline/question_parsing/question_parsing.py).

## Sortie

**Toujours une `list[dict]`** (drop-in compat avec `src.question.understand_question`). Chaque entrée :

| Section | Contenu |
|---|---|
| `retrieval`  | `main_query`, `page_hint`, `section_hint`, `layout_hint`, `document_hint`, `anchor_keywords` |
| `generation` | `original_question`, `format_constraint`, `disambiguation`, `must_distinguish` |
| `_meta`      | `intent`, `document_type`, `bricks_active` (traçabilité) |

**Le JSON ne contient QUE des champs populés** (pas de `null`).

## Connaissance métier dans les PRESETS

| `document_type` | Briques actives |
|---|---|
| `pdf`   | toutes |
| `word`  | sans `page_hint` (pas de pagination stable en .docx) |
| `excel` | sans `page_hint` ni `section_hint` |
| `email` | sans `page_hint` ni `section_hint` |
| `pptx`  | toutes (slide = page) |

Override : `parse_question(q, enable={"page_hint": False})`.

## Approche

Regex / heuristique uniquement (pas de LLM, conforme à `CLAUDE.md`).

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## 1 — Démonstration sur 8 questions variées

On lance `parse_question` sur des questions couvrant tous les cas d'usage : un seul hint, hints multiples, question composée, disambiguation, format constraint, codes juridiques, question minimale.

In [2]:
import json
from docpipeline.question_parsing import parse_question

QUESTIONS = [
    "Quelle est la date d'effet du contrat ? Page 1, format YYYY-MM-DD.",
    "What is the limit per claim, not the deductible, in the Limits section page 5? Article L131-1 applies.",
    "Look in the exclusions section for the flooding clause.",
    "Compare the indemnification and liability caps in the latest version.",
    "List all the obligations of the seller, in bullet list.",
    "Find the policy number - it's usually in the header, format JSON.",
    "Dans le tableau page 3 du contrat MAIF, quel est le montant ? En euros.",
    "Combien coute l'assurance ?",
]

for q in QUESTIONS:
    plan = parse_question(q, document_type='pdf')
    print(f'Q: {q}')
    print(json.dumps(plan, indent=2, ensure_ascii=False))
    print('-' * 80)

Q: Quelle est la date d'effet du contrat ? Page 1, format YYYY-MM-DD.
[
  {
    "retrieval": {
      "main_query": "Quelle est la date d'effet du contrat ? Page 1, format YYYY-MM-DD.",
      "page_hint": 1
    },
    "generation": {
      "original_question": "Quelle est la date d'effet du contrat ? Page 1, format YYYY-MM-DD.",
      "format_constraint": "ISO 8601 date (YYYY-MM-DD)"
    },
    "_meta": {
      "intent": "extract",
      "document_type": "pdf",
      "bricks_active": [
        "page_hint",
        "format"
      ]
    }
  }
]
--------------------------------------------------------------------------------
Q: What is the limit per claim, not the deductible, in the Limits section page 5? Article L131-1 applies.
[
  {
    "retrieval": {
      "main_query": "What is the limit per claim, not the deductible, in the Limits section page 5? Article L131-1 applies.",
      "anchor_keywords": [
        "L131-1",
        "Article L131-1"
      ],
      "page_hint": 5,
      "sectio

### Lecture des résultats

**Question 1** — `Quelle est la date d'effet ... Page 1, format YYYY-MM-DD` :
- `retrieval.page_hint = 1` → le retrieval ira chercher d'abord en page 1.
- `generation.format_constraint = ISO 8601 date (YYYY-MM-DD)` → la génération sait que la réponse doit être au format ISO.
- `_meta.bricks_active = [page_hint, format]` → exactement les briques attendues.
- JSON propre, pas de `null`.

**Question 2** — `What is the limit per claim, not the deductible ... section page 5? Article L131-1` (la plus riche) :
- `retrieval.page_hint = 5`, `retrieval.section_hint = limits` → double filtre.
- `retrieval.anchor_keywords` contient `L131-1` → BM25 boostera sur ce code juridique.
- `generation.must_distinguish = [deductible]` → l'instruction de désambiguïsation va vers la génération (pas le retrieval, sinon on supprimerait la ligne qui contient le plafond).
- 5 briques actives : anchor_keywords, page_hint, section_hint, format, disambiguation.

**Question 3** — `Look in the exclusions section ...` (cas piégeux) :
- `retrieval.section_hint = exclusions` → la regex capture bien le nom de section.
- **Bug Kezhan corrigé** : la version `src/question/` retournait `for the flooding clause` (regex trop greedy).

**Question 4** — `Compare the indemnification and liability caps in the latest version` :
- `_meta.intent = compare` → le pipeline aval saura router vers une stratégie comparative.
- `retrieval.document_hint = latest version` → filtre de version.
- Note : la décomposition compound (en sous-questions) n'est pas implémentée ici (LLM-only chez Kezhan).

**Question 5** — `List all the obligations ... in bullet list` :
- `_meta.intent = aggregate` → stratégie listing exhaustive.
- `generation.format_constraint = bullet list`.

**Question 6** — `Find the policy number ... in the header, format JSON` :
- `retrieval.layout_hint = header` → on cherchera dans les zones d'en-tête.
- `generation.format_constraint = valid JSON`.

**Question 7** — `Dans le tableau page 3 du contrat MAIF ... En euros` (français riche) :
- `retrieval.page_hint = 3`, `retrieval.layout_hint = table`, `retrieval.anchor_keywords` contient `MAIF`.
- `generation.format_constraint = numeric value in EUR`.

**Question 8** — `Combien coute l'assurance ?` (minimale) :
- Aucun hint structurel détecté → `retrieval` ne contient que `main_query`, `_meta.bricks_active = []`.
- Pas de `null` dans le JSON malgré l'absence de hints.

## 2 — Banque de cas (vérification des correctifs vs `src/question/`)

Couvre 7 bugs / faux positifs de la version `src/question/` initiale qu'on a corrigés ici.

In [3]:
from docpipeline.question_parsing import parse_question
import json

CHECKS = [
    {
        'q': 'Look in the exclusions section for the flooding clause.',
        'doc': 'pdf',
        'check': lambda p: p[0]['retrieval'].get('section_hint') == 'exclusions',
        'desc': 'section_hint = "exclusions" (pas "for the flooding clause")',
    },
    {
        'q': 'Does article L131-1 of the insurance code apply here?',
        'doc': 'pdf',
        'check': lambda p: any('L131-1' in k or '131-1' in k for k in p[0]['retrieval'].get('anchor_keywords', [])),
        'desc': 'anchor_keywords contient L131-1',
    },
    {
        'q': 'Date au format YYYY-MM-DD',
        'doc': 'pdf',
        'check': lambda p: 'YYYY' not in p[0]['retrieval'].get('anchor_keywords', []),
        'desc': 'YYYY exclu des anchor_keywords (faux positif evite)',
    },
    {
        'q': 'What is the maximum coverage amount?',
        'doc': 'pdf',
        'check': lambda p: p[0]['_meta']['intent'] == 'extract',
        'desc': 'intent = extract (pas yes_no)',
    },
    {
        'q': 'Plafond page 3 du contrat',
        'doc': 'word',
        'check': lambda p: 'page_hint' not in p[0]['retrieval'],
        'desc': 'En Word, page_hint ignore (preset Word sans page_hint)',
    },
    {
        'q': 'Le plafond, pas la franchise',
        'doc': 'pdf',
        'check': lambda p: any('franchise' in d for d in p[0]['generation'].get('must_distinguish', [])),
        'desc': 'disambiguation detecte "franchise" comme distractor',
    },
    {
        'q': "Combien coute l'assurance ?",
        'doc': 'pdf',
        'check': lambda p: 'null' not in json.dumps(p),
        'desc': 'JSON sans null sur question minimale',
    },
]

for c in CHECKS:
    plan = parse_question(c['q'], document_type=c['doc'])
    ok = c['check'](plan)
    flag = 'OK  ' if ok else 'FAIL'
    print(f"[{flag}] {c['desc']}")
    print(f"       Q: {c['q']}  (doc_type={c['doc']})")

[OK  ] section_hint = "exclusions" (pas "for the flooding clause")
       Q: Look in the exclusions section for the flooding clause.  (doc_type=pdf)
[OK  ] anchor_keywords contient L131-1
       Q: Does article L131-1 of the insurance code apply here?  (doc_type=pdf)
[OK  ] YYYY exclu des anchor_keywords (faux positif evite)
       Q: Date au format YYYY-MM-DD  (doc_type=pdf)
[OK  ] intent = extract (pas yes_no)
       Q: What is the maximum coverage amount?  (doc_type=pdf)
[OK  ] En Word, page_hint ignore (preset Word sans page_hint)
       Q: Plafond page 3 du contrat  (doc_type=word)
[OK  ] disambiguation detecte "franchise" comme distractor
       Q: Le plafond, pas la franchise  (doc_type=pdf)
[OK  ] JSON sans null sur question minimale
       Q: Combien coute l'assurance ?  (doc_type=pdf)


### Lecture des résultats

Les 7 checks doivent tous être `OK`. Chacun couvre un correctif sur la version `src/question/` initiale :

| # | Bug `src/question/` | Cas testé | Correctif `docpipeline.question_parsing` |
|---|---|---|---|
| 1 | Regex `section\s+([\w\s]+?)` trop greedy | « in the exclusions section for the flooding clause » | Ajout pattern « X section » (nom AVANT « section ») non greedy |
| 2 | Pas de pattern article juridique | « Article L131-1 » | Ajout `re.compile(r'\b[lLrR]\.?\s?\d+[-]\d+\b')` |
| 3 | `YYYY` détecté comme code | « format YYYY-MM-DD » | `YYYY/MM/DD/JSON/...` ajoutés à `_ANCHOR_STOPWORDS` |
| 4 | « what is » classé `yes_no` | « What is the maximum ... » | Réordonné : `extract` testé avant `yes_no` |
| 5 | `page_hint` actif sur Word | doc_type=word + « page 3 » | Preset `word` ne contient pas `page_hint` |
| 6 | Disambiguation FR partielle | « pas la franchise » | Pattern français ajouté |
| 7 | (pas un bug, garde-fou) | Question minimale | Vérifie qu'aucun `null` ne pollue le JSON |

Si un seul `FAIL` apparaît, c'est qu'un correctif a régressé.

## 3 — Document-aware presets

Même question, cinq types de documents : on vérifie que le preset adapte les briques actives selon le `document_type`.

In [4]:
from docpipeline.question_parsing import parse_question
import json

Q = 'Le plafond dans la section Risques, page 3, en euros'

for doc_type in ('pdf', 'word', 'excel', 'email', 'pptx'):
    plan = parse_question(Q, document_type=doc_type)
    print(f'=== document_type = {doc_type} ===')
    print(json.dumps(plan[0], indent=2, ensure_ascii=False))
    print()

=== document_type = pdf ===
{
  "retrieval": {
    "main_query": "Le plafond dans la section Risques, page 3, en euros",
    "page_hint": 3,
    "section_hint": "risques"
  },
  "generation": {
    "original_question": "Le plafond dans la section Risques, page 3, en euros",
    "format_constraint": "numeric value in EUR"
  },
  "_meta": {
    "intent": "extract",
    "document_type": "pdf",
    "bricks_active": [
      "page_hint",
      "section_hint",
      "format"
    ]
  }
}

=== document_type = word ===
{
  "retrieval": {
    "main_query": "Le plafond dans la section Risques, page 3, en euros",
    "section_hint": "risques"
  },
  "generation": {
    "original_question": "Le plafond dans la section Risques, page 3, en euros",
    "format_constraint": "numeric value in EUR"
  },
  "_meta": {
    "intent": "extract",
    "document_type": "word",
    "bricks_active": [
      "section_hint",
      "format"
    ]
  }
}

=== document_type = excel ===
{
  "retrieval": {
    "main_query"

### Lecture des résultats

La même question donne des outputs **différents** selon le `document_type` :

| `document_type` | `page_hint` présent ? | `section_hint` présent ? | Justification |
|---|---|---|---|
| `pdf`   | oui | oui | PDF : pages stables + sections déclarables |
| `word`  | **non** | oui | .docx : pagination dépend du renderer (font, marges, écran) |
| `excel` | non | **non** | xlsx : pas de pagination ni de TOC, juste feuilles + colonnes |
| `email` | non | non | .eml : ni pagination ni section formelle |
| `pptx`  | oui (= numéro de slide) | oui | slides = pages, groupes = sections |

C'est la **connaissance métier centralisée dans les PRESETS** : on évite que le retriever en aval reçoive un faux signal `page_hint=3` sur un .docx où la « page 3 » n'a pas de sens stable.

## 4 — Override fin via `enable={...}`

Le preset est un défaut. À l'appel, on peut désactiver ou activer brique par brique.

In [5]:
from docpipeline.question_parsing import parse_question
import json

Q = 'Le plafond page 3, format JSON'

print('=== Sans override (preset PDF complet) ===')
print(json.dumps(parse_question(Q)[0]['_meta']['bricks_active'], ensure_ascii=False))

print('\n=== enable={"page_hint": False} (on coupe page_hint) ===')
print(json.dumps(parse_question(Q, enable={'page_hint': False})[0]['_meta']['bricks_active'], ensure_ascii=False))

print('\n=== Word + force page_hint=True (override preset) ===')
print(json.dumps(parse_question(Q, document_type='word', enable={'page_hint': True})[0]['_meta']['bricks_active'], ensure_ascii=False))

=== Sans override (preset PDF complet) ===
["page_hint", "format"]

=== enable={"page_hint": False} (on coupe page_hint) ===
["format"]

=== Word + force page_hint=True (override preset) ===
["format"]


### Lecture des résultats

1. **Sans override** : le preset PDF active toutes les briques compatibles. Sur cette question, `page_hint` et `format` font hit.
2. **`enable={"page_hint": False}`** : `page_hint` retiré du run, même sur PDF. Utile quand on sait que la question est triviale et qu'on veut économiser.
3. **Word + `enable={"page_hint": True}`** : on force `page_hint` même sur Word (le preset ne l'inclut pas). Cas d'usage rare mais utile pour debug.

Convention : `enable={X: False}` retire X du preset, `enable={Y: True}` ajoute Y au preset.